# 19 Combine 46 raw stock features

目标：在统一普通股票基础宇宙内汇合 6+30+10 共 46 个原始股票特征。

- 输入：14、16、17、18 号输出
- 正式输出：stock_46_raw_features.parquet、特征覆盖审计
- 主键：permno+month
- 上游依赖：14、16、17、18
- smoke 模式仅使用真实本地数据子样本，并写入 `data/audit/smoke/`；不冒充正式结果。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import os
import duckdb
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic
from src.runtime_paths import output_path, upstream_path

def esc(path): return str(path).replace("'","''")
base_path=upstream_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/stock_monthly_clean.parquet")
ret_path=upstream_path(ROOT,RUN_MODE,"16_compute_stock_return_features","data/processed/stock/stock_return_features_raw.parquet")
acc_path=upstream_path(ROOT,RUN_MODE,"17_compute_stock_accounting_features","data/processed/stock/stock_accounting_features_raw.parquet")
risk_path=upstream_path(ROOT,RUN_MODE,"18_compute_stock_risk_liquidity_features","data/processed/stock/stock_risk_liquidity_features_raw.parquet")
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET preserve_insertion_order=false")
names=['r2_1', 'r12_2', 'r12_7', 'r36_13', 'ST_Rev', 'LT_Rev', 'Investment', 'NOA', 'DPI2A', 'NI', 'PROF', 'ATO', 'CTO', 'FC2Y', 'OP', 'PM', 'RNA', 'ROA', 'ROE', 'SGA2S', 'D2A', 'AC', 'OA', 'OL', 'PCM', 'A2ME', 'BEME', 'C', 'CF', 'CF2P', 'D2P', 'E2P', 'Q', 'S2P', 'Lev', 'AT', 'Beta', 'IdioVol', 'LME', 'LTurnover', 'MktBeta', 'Rel2High', 'Resid_Var', 'Spread', 'SUV', 'Variance']
return_names=['r2_1', 'r12_2', 'r12_7', 'r36_13', 'ST_Rev', 'LT_Rev']
accounting_names=['Investment', 'NOA', 'DPI2A', 'NI', 'PROF', 'ATO', 'CTO', 'FC2Y', 'OP', 'PM', 'RNA', 'ROA', 'ROE', 'SGA2S', 'D2A', 'AC', 'OA', 'OL', 'PCM', 'A2ME', 'BEME', 'C', 'CF', 'CF2P', 'D2P', 'E2P', 'Q', 'S2P', 'Lev', 'AT']
risk_names=['Beta', 'IdioVol', 'LME', 'LTurnover', 'MktBeta', 'Rel2High', 'Resid_Var', 'Spread', 'SUV', 'Variance']
assert len(names)==46 and len(set(names))==46
def qcols(alias,columns): return ",".join(f'{alias}."{c}"' for c in columns)
required_historical_keys="""(
   (b.permno=18307 AND b."month"=DATE '1931-06-01')
   OR (b.permno=22736 AND b."month"=DATE '1961-03-01')
   OR (b.permno=76154 AND b."month"=DATE '1966-05-01')
 )"""
panel_query=f"""SELECT b.permno,b."month",b.mthret,b.market_equity_thousand,b.common_stock,
 {qcols("r",return_names)},{qcols("a",accounting_names)},{qcols("k",risk_names)}
 FROM read_parquet('{esc(base_path)}') b
 LEFT JOIN read_parquet('{esc(ret_path)}') r USING(permno,"month")
 LEFT JOIN read_parquet('{esc(acc_path)}') a USING(permno,"month")
 LEFT JOIN read_parquet('{esc(risk_path)}') k USING(permno,"month")
 WHERE b.common_stock OR {required_historical_keys}"""
out=output_path(ROOT,RUN_MODE,"19_combine_46_raw_stock_features","data/processed/stock/stock_46_raw_features.parquet")
aud1=output_path(ROOT,RUN_MODE,"19_combine_46_raw_stock_features","data/audit/19_feature_coverage.csv")
aud2=output_path(ROOT,RUN_MODE,"19_combine_46_raw_stock_features","data/audit/19_join_audit.csv")
staged=PATHS["full_run_tmp"]/"19_stock_46_raw_staged.parquet"
staged.unlink(missing_ok=True)
duckdb_copy_parquet_atomic(con,panel_query,staged)
con.execute(f"CREATE TEMP VIEW panel AS SELECT * FROM read_parquet('{esc(staged)}')")
coverage_expr=",".join(
    f'count("{c}") n_{c}' for c in names
)
duckdb_copy_csv_atomic(con,f"""SELECT "month",count(*) n_stocks,{coverage_expr}
 FROM panel GROUP BY "month" ORDER BY "month" """,aud1)
base_count=int(con.execute(f"""SELECT count(*) FROM read_parquet('{esc(base_path)}') b
 WHERE b.common_stock OR {required_historical_keys}""").fetchone()[0])
ret_count=int(con.execute(f"SELECT count(*) FROM read_parquet('{esc(ret_path)}')").fetchone()[0])
acc_count=int(con.execute(f"SELECT count(*) FROM read_parquet('{esc(acc_path)}')").fetchone()[0])
risk_count=int(con.execute(f"SELECT count(*) FROM read_parquet('{esc(risk_path)}')").fetchone()[0])
panel_count=int(con.execute("SELECT count(*) FROM panel").fetchone()[0])
expansion_ratio=panel_count/base_count if base_count else 1.0
duckdb_copy_csv_atomic(con,f"""SELECT * FROM (VALUES
 ('16_compute_stock_return_features',{base_count},{ret_count},{panel_count},{expansion_ratio}),
 ('17_compute_stock_accounting_features',{base_count},{acc_count},{panel_count},{expansion_ratio}),
 ('18_compute_stock_risk_liquidity_features',{base_count},{risk_count},{panel_count},{expansion_ratio})
 ) t(source,left_rows,right_rows,result_rows,expansion_ratio)""",aud2)
duplicate_count=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM panel GROUP BY permno,"month" HAVING count(*)>1)""").fetchone()[0])
missing_base_keys=int(con.execute(f"""SELECT count(*) FROM (
 SELECT b.permno,b."month" FROM read_parquet('{esc(base_path)}') b
 WHERE b.common_stock OR {required_historical_keys}
 EXCEPT SELECT permno,"month" FROM panel)""").fetchone()[0])
extra_base_keys=int(con.execute(f"""SELECT count(*) FROM (
 SELECT permno,"month" FROM panel
 EXCEPT SELECT b.permno,b."month" FROM read_parquet('{esc(base_path)}') b
 WHERE b.common_stock OR {required_historical_keys})""").fetchone()[0])
required_historical_count=int(con.execute("""SELECT count(*) FROM panel WHERE
 (permno=18307 AND "month"=DATE '1931-06-01')
 OR (permno=22736 AND "month"=DATE '1961-03-01')
 OR (permno=76154 AND "month"=DATE '1966-05-01')""").fetchone()[0])
required_historical_base_count=int(con.execute(f"""SELECT count(*)
 FROM read_parquet('{esc(base_path)}') b
 WHERE {required_historical_keys}""").fetchone()[0])
columns=[r[0] for r in con.execute("DESCRIBE panel").fetchall()]
assert columns==["permno","month","mthret","market_equity_thousand","common_stock"]+names
assert duplicate_count==0 and panel_count==base_count
assert missing_base_keys==0 and extra_base_keys==0
if RUN_MODE=="full":
    assert required_historical_count==required_historical_base_count
con.close()
out.parent.mkdir(parents=True,exist_ok=True)
os.replace(staged,out)
INPUTS_USED=["16_compute_stock_return_features","17_compute_stock_accounting_features","18_compute_stock_risk_liquidity_features","stock_monthly_clean.parquet"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,aud1,aud2]]
INPUT_COUNTS={"base_common_stock":base_count,"required_historical_keys_in_conflict_free_base":required_historical_base_count}; OUTPUT_COUNTS={"raw_46":panel_count,"required_historical_keys_retained":required_historical_count}; PRIMARY_KEY=["permno","month"]; DUPLICATE_KEY_COUNT=duplicate_count
EXCLUDED_ROW_COUNT=0; EXCEPTION_FILE=str(aud2.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["Exactly 46 named raw stock features are present.","The stage-19 key set exactly equals its locked ordinary-stock base universe and retains every canonical early key that survives Notebook 14's mandatory whole-key conflict exclusion.","The canonical early-key audit covers 18307/1931-06, 22736/1961-03 and 76154/1966-05 without overriding Notebook 14 conflicts.","No artificial 1967 start-date restriction is applied; the universe is never narrowed to the mutual-fund holdings subset."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "19_combine_46_raw_stock_features", stage="19", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
